# 04_01 – Постановка задачи и состав наблюдения

Работаем с Европой 2019–2023. Вход – последняя проверенная копия после подготовки 03. Здесь определяем задачу исследования; строки и значения не меняем.

Рабочая единица наблюдения – строка набора, связанная с конечной посадкой и обнаруженными попытками. Она не равнозначна отдельной попытке или гарантированно одному эксплуатационному уходу. Повторяющийся flight_id не является самостоятельным уникальным ключом. GoA отражает наличие обнаруженного ухода в исходной разметке; независимая траекторная валидация остается отдельной задачей. [Описание источника](https://zenodo.org/records/10210178).

Цель первого эксперимента предлагается сформулировать как оценку вероятности GoA=True по информации, доступной до первого наблюдаемого захода. Согласовано: все наблюдаемые полеты, включая специальные; прогноз в точке входа C40. Проверяем сопоставление и влияние пропусков до отбора.

In [1]:
# Подключаем инструменты чтения и проверки целостности
from pathlib import Path
import hashlib
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Находим корень проекта из любой папки запуска
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
INPUT = ROOT / 'europe/data/working/03_c40_diagnostic_flags.parquet'
REPORT = ROOT / 'europe/reports/preparation'
journal = json.loads((REPORT / '07_c40_flags_applied.json').read_text(encoding='utf8'))

# Сверяем рабочую копию с журналом согласованного преобразования
with INPUT.open('rb') as stream:
    input_hash = hashlib.file_digest(stream, 'sha256').hexdigest()
assert input_hash == journal['output_sha256']
reader = pq.ParquetFile(INPUT)
assert reader.metadata.num_rows == journal['rows_after']
display(pd.DataFrame([{'input': str(INPUT.relative_to(ROOT)),
                      'rows': reader.metadata.num_rows,
                      'go_arounds': journal['go_arounds_after'],
                      'sha256': input_hash}]))

,input,rows,go_arounds,sha256
0,europe\data\working\03_c40_diagnostic_flags.pa...,7984254,26253,ddf11438bab44137242ffcf4da928418e116efd46defce...


## Момент прогноза определяет допустимые данные

Вариант A – первая пригодная отметка C40 до первой попытки. Нужны проверка соответствия отметки наблюдению и анализ потерь из-за отсутствующего/позднего времени. Сами геометрические поля C40 не подтверждают корректность присоединения.

Вариант B – фиксированный момент до начала первой попытки. Для признаков в этот момент могут понадобиться исходные траектории и исторические погодные наблюдения. Одной итоговой таблицы недостаточно, чтобы восстановить все динамические признаки.

Для прогноза в точке C40 сначала оцениваем пропуски; пока не исключаем отсутствующие C40 и не объявляем погодные поля доступными. Итоговые характеристики посадки и обнаруженных попыток нельзя автоматически включать в прогноз до события.

In [2]:
# Показываем фактическую схему входа перед обсуждением признаков
display(pd.DataFrame([{'field': field.name, 'type': str(field.type)}
                      for field in reader.schema_arrow]))

# Фиксируем предложения и открытые вопросы вместо скрытых правил фильтрации
decisions = pd.DataFrame([
    ('Период', 'Европа 2019–2023', 'Принято'),
    ('Единица наблюдения', 'Строка с конечной посадкой; не отдельная попытка', 'Рабочее определение'),
    ('Целевая метка', 'GoA исходной алгоритмической разметки', 'Ограничение источника'),
    ('Популяция', 'Все наблюдаемые полеты, включая специальные; сравнение групп отдельно', 'Принято'),
    ('Момент прогноза', 'Вход C40; проверяем сопоставление и пропуски', 'Принято'),
    ('Несколько попыток', 'Не исключать только по числу попыток', 'Принято'),
    ('Специальные полеты', 'Подтвердить назначение и оценить влияние исключений', 'Отложено'),
    ('Аэропорты', 'Проверить покрытие, качество и размер целевого класса', 'Отложено'),
    ('Разделение выборки', 'Временной тест и отдельная проверка переноса между аэропортами', 'Предложение'),
], columns=['question', 'definition_or_action', 'status'])
display(decisions)

,field,type
0,flight_id,string
1,callsign,string
2,icao24,string
3,landing_time,"timestamp[ns, tz=UTC]"
4,registration,string
5,typecode,string
6,icaoaircrafttype,string
7,airport,string
8,country,string
9,attempt_times,"list<element: timestamp[us, tz=UTC]>"


,question,definition_or_action,status
0,Период,Европа 2019–2023,Принято
1,Единица наблюдения,Строка с конечной посадкой; не отдельная попытка,Рабочее определение
2,Целевая метка,GoA исходной алгоритмической разметки,Ограничение источника
3,Популяция,"Все наблюдаемые полеты, включая специальные; с...",Принято
4,Момент прогноза,Вход C40; проверяем сопоставление и пропуски,Принято
5,Несколько попыток,Не исключать только по числу попыток,Принято
6,Специальные полеты,Подтвердить назначение и оценить влияние исклю...,Отложено
7,Аэропорты,"Проверить покрытие, качество и размер целевого...",Отложено
8,Разделение выборки,Временной тест и отдельная проверка переноса м...,Предложение


## Следующий шаг

04_02 – карта доступности признаков: время измерения, момент присоединения, пропуски, происхождение и связь с целевой меткой. Затем оцениваем предложенные правила отбора на полной базе, показывая число затронутых строк и GoA=True. Фильтрация требует обоснованного решения по конкретному правилу.

Метаданные источника, идентификаторы и диагностические флаги храним для аудита. Их наличие в таблице не делает их признаками модели. Статистики будущего тестового периода не используем для обучения преобразований.

История подготовки и отложенных вопросов: [карта 03](../03_data_preparation/README.md), [словарь полей](../../docs/data_dictionary.md).